# Project 2: Market-Basket Analysis on IMDb Movie Actors

This notebook implements Project 2 using the IMDb Movies Dataset from Kaggle.
Each movie is treated as one basket, and the actors in `Star1`, `Star2`, `Star3`, and `Star4` are treated as items.

The main method is the Apriori algorithm. The notebook finds frequent actor itemsets and association rules, and also compares different support/confidence thresholds.

## 1. Install Dependencies

Run this cell in Google Colab before the rest of the notebook.

In [ ]:
!pip install -q pandas kaggle

## 2. Kaggle Authentication and Dataset Download

Before running this cell, replace `xxxxxx` with your Kaggle username and API key. Before uploading this notebook publicly, change them back to `xxxxxx`.

Dataset identifier:
`harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows`

In [ ]:
import os
from pathlib import Path

KAGGLE_USERNAME = "xxxxxx"
KAGGLE_KEY = "xxxxxx"

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

DATASET_ID = "harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows"
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
OUTPUT_DIR = BASE_DIR / "outputs"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if KAGGLE_USERNAME == "xxxxxx" or KAGGLE_KEY == "xxxxxx":
    print("Please replace KAGGLE_USERNAME and KAGGLE_KEY with your Kaggle API credentials before downloading.")
else:
    !kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows -p data/raw --unzip

## 3. Imports and Global Settings

In [ ]:
from __future__ import annotations

import itertools
import math
import re
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Set, Tuple

import pandas as pd

STAR_COLUMNS = ["Star1", "Star2", "Star3", "Star4"]

DEFAULT_MIN_SUPPORT = 0.003
DEFAULT_MIN_CONFIDENCE = 0.50
DEFAULT_MAX_ITEMSET_SIZE = 4

USE_SUBSAMPLE = False
SUBSAMPLE_SIZE = 200
RANDOM_STATE = 42

THRESHOLD_EXPERIMENTS = [
    ("High support", 0.020, 0.50),
    ("Loose", 0.002, 0.20),
    ("Final", DEFAULT_MIN_SUPPORT, DEFAULT_MIN_CONFIDENCE),
]

Itemset = Tuple[str, ...]
Transaction = Set[str]

## 4. Load and Pre-process the Dataset

The pre-processing keeps only the four actor columns, removes missing or empty actor names, normalizes spacing and Unicode formatting, and stores each movie as a set of actors.

In [ ]:
def find_dataset_csv(raw_data_dir: Path = RAW_DATA_DIR) -> Path:
    csv_files = sorted(raw_data_dir.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(
            f"No CSV file found under {raw_data_dir.resolve()}. "
            "Run the Kaggle download cell first."
        )

    for csv_path in csv_files:
        try:
            columns = pd.read_csv(csv_path, nrows=0).columns
        except Exception:
            continue
        if all(column in columns for column in STAR_COLUMNS):
            return csv_path

    raise FileNotFoundError(
        "A CSV file was found, but none contained all required columns: "
        + ", ".join(STAR_COLUMNS)
    )


def normalize_actor_name(value: object) -> str | None:
    if pd.isna(value):
        return None

    name = unicodedata.normalize("NFKC", str(value))
    name = name.replace("\u00a0", " ")
    name = re.sub(r"\s+", " ", name)
    name = name.strip(" ,;")

    if not name:
        return None

    return name


def row_to_transaction(row: pd.Series) -> Transaction:
    actors: Transaction = set()
    for column in STAR_COLUMNS:
        actor = normalize_actor_name(row.get(column))
        if actor:
            actors.add(actor)
    return actors


def load_transactions(
    csv_path: Path,
    use_subsample: bool = USE_SUBSAMPLE,
    subsample_size: int = SUBSAMPLE_SIZE,
) -> List[Transaction]:
    df = pd.read_csv(csv_path, usecols=STAR_COLUMNS, dtype=str)

    if use_subsample:
        if subsample_size <= 0:
            raise ValueError("subsample_size must be positive.")
        sample_size = min(subsample_size, len(df))
        df = df.sample(n=sample_size, random_state=RANDOM_STATE)

    transactions = [row_to_transaction(row) for _, row in df.iterrows()]
    transactions = [transaction for transaction in transactions if transaction]

    if not transactions:
        raise ValueError("No valid actor baskets were created from the dataset.")

    return transactions

## 5. Apriori Implementation

In [ ]:
def minimum_support_count(min_support: float, n_transactions: int) -> int:
    if min_support <= 0:
        raise ValueError("Minimum support must be positive.")
    if min_support <= 1:
        return max(1, math.ceil(min_support * n_transactions))
    return int(math.ceil(min_support))


def count_singletons(transactions: Sequence[Transaction]) -> Counter[Itemset]:
    counts: Counter[Itemset] = Counter()
    for transaction in transactions:
        for item in transaction:
            counts[(item,)] += 1
    return counts


def filter_frequent(counts: Counter[Itemset], min_count: int) -> Dict[Itemset, int]:
    return {itemset: count for itemset, count in counts.items() if count >= min_count}


def generate_candidates(previous_frequents: Iterable[Itemset], k: int) -> Set[Itemset]:
    previous_set = set(previous_frequents)
    previous_list = sorted(previous_set)
    candidates: Set[Itemset] = set()

    for left_index in range(len(previous_list)):
        for right_index in range(left_index + 1, len(previous_list)):
            union = tuple(sorted(set(previous_list[left_index]) | set(previous_list[right_index])))
            if len(union) != k:
                continue

            all_subsets_are_frequent = all(
                tuple(sorted(subset)) in previous_set
                for subset in itertools.combinations(union, k - 1)
            )
            if all_subsets_are_frequent:
                candidates.add(union)

    return candidates


def count_candidates(
    transactions: Sequence[Transaction],
    candidates: Set[Itemset],
    k: int,
) -> Counter[Itemset]:
    counts: Counter[Itemset] = Counter()
    if not candidates:
        return counts

    for transaction in transactions:
        if len(transaction) < k:
            continue
        for combination in itertools.combinations(sorted(transaction), k):
            if combination in candidates:
                counts[combination] += 1

    return counts


def apriori(
    transactions: Sequence[Transaction],
    min_support: float,
    max_itemset_size: int = DEFAULT_MAX_ITEMSET_SIZE,
) -> Tuple[Dict[int, Dict[Itemset, int]], Dict[Itemset, int], int]:
    if max_itemset_size < 1:
        raise ValueError("max_itemset_size must be at least 1.")

    n_transactions = len(transactions)
    min_count = minimum_support_count(min_support, n_transactions)

    frequent_by_size: Dict[int, Dict[Itemset, int]] = {}
    all_support_counts: Dict[Itemset, int] = {}

    frequent_1 = filter_frequent(count_singletons(transactions), min_count)
    frequent_by_size[1] = frequent_1
    all_support_counts.update(frequent_1)

    previous_frequents = set(frequent_1)

    for k in range(2, max_itemset_size + 1):
        candidates = generate_candidates(previous_frequents, k)
        candidate_counts = count_candidates(transactions, candidates, k)
        frequent_k = filter_frequent(candidate_counts, min_count)

        if not frequent_k:
            break

        frequent_by_size[k] = frequent_k
        all_support_counts.update(frequent_k)
        previous_frequents = set(frequent_k)

    return frequent_by_size, all_support_counts, min_count

## 6. Association Rules and Output Helpers

In [ ]:
def support_fraction(count: int, n_transactions: int) -> float:
    return count / n_transactions


def generate_association_rules(
    all_support_counts: Dict[Itemset, int],
    n_transactions: int,
    min_confidence: float,
) -> pd.DataFrame:
    rules = []

    for itemset, itemset_count in all_support_counts.items():
        if len(itemset) < 2:
            continue

        itemset_items = set(itemset)
        for antecedent_size in range(1, len(itemset)):
            for antecedent_tuple in itertools.combinations(itemset, antecedent_size):
                antecedent = tuple(sorted(antecedent_tuple))
                consequent = tuple(sorted(itemset_items - set(antecedent)))

                antecedent_count = all_support_counts.get(antecedent)
                consequent_count = all_support_counts.get(consequent)
                if not antecedent_count or not consequent_count:
                    continue

                confidence = itemset_count / antecedent_count
                if confidence < min_confidence:
                    continue

                consequent_support = support_fraction(consequent_count, n_transactions)
                lift = confidence / consequent_support if consequent_support else float("inf")

                rules.append(
                    {
                        "antecedent": " | ".join(antecedent),
                        "consequent": " | ".join(consequent),
                        "support_count": itemset_count,
                        "support": support_fraction(itemset_count, n_transactions),
                        "confidence": confidence,
                        "lift": lift,
                    }
                )

    rules_df = pd.DataFrame(
        rules,
        columns=["antecedent", "consequent", "support_count", "support", "confidence", "lift"],
    )
    if rules_df.empty:
        return rules_df

    return rules_df.sort_values(
        by=["confidence", "lift", "support_count"],
        ascending=[False, False, False],
    ).reset_index(drop=True)


def frequent_itemsets_to_dataframe(
    frequent_by_size: Dict[int, Dict[Itemset, int]],
    n_transactions: int,
) -> pd.DataFrame:
    rows = []
    for size, itemsets in frequent_by_size.items():
        for itemset, count in itemsets.items():
            rows.append(
                {
                    "size": size,
                    "itemset": " | ".join(itemset),
                    "support_count": count,
                    "support": support_fraction(count, n_transactions),
                }
            )

    result = pd.DataFrame(rows, columns=["size", "itemset", "support_count", "support"])
    if result.empty:
        return result

    return result.sort_values(
        by=["size", "support_count", "itemset"],
        ascending=[True, False, True],
    ).reset_index(drop=True)

## 7. Load Transactions

In [ ]:
csv_path = find_dataset_csv()
print(f"Using dataset file: {csv_path}")

transactions = load_transactions(csv_path)
unique_actors = sorted({actor for transaction in transactions for actor in transaction})
basket_sizes = [len(transaction) for transaction in transactions]

print(f"Movies/baskets analyzed: {len(transactions)}")
print(f"Unique actors/items: {len(unique_actors)}")
print(f"Average basket size: {sum(basket_sizes) / len(basket_sizes):.2f}")

## 8. Threshold Comparison Experiment

This experiment compares three support/confidence settings and saves `outputs/threshold_comparison.csv`.

In [ ]:
def run_threshold_comparison(transactions: Sequence[Transaction]) -> pd.DataFrame:
    rows = []
    n_transactions = len(transactions)

    for setting_name, min_support, min_confidence in THRESHOLD_EXPERIMENTS:
        frequent_by_size, all_support_counts, min_count = apriori(
            transactions=transactions,
            min_support=min_support,
            max_itemset_size=DEFAULT_MAX_ITEMSET_SIZE,
        )
        rules_df = generate_association_rules(
            all_support_counts=all_support_counts,
            n_transactions=n_transactions,
            min_confidence=min_confidence,
        )

        row = {
            "setting": setting_name,
            "min_support": min_support,
            "min_confidence": min_confidence,
            "min_support_count": min_count,
            "frequent_itemsets": sum(len(itemsets) for itemsets in frequent_by_size.values()),
            "association_rules": len(rules_df),
        }
        row.update({f"frequent_size_{size}": len(itemsets) for size, itemsets in frequent_by_size.items()})
        rows.append(row)

    comparison_df = pd.DataFrame(rows).fillna(0)
    comparison_df.to_csv(OUTPUT_DIR / "threshold_comparison.csv", index=False)
    return comparison_df

threshold_comparison_df = run_threshold_comparison(transactions)
threshold_comparison_df

## 9. Final Experiment

The final setting uses `min_support = 0.003` and `min_confidence = 0.50`.

In [ ]:
frequent_by_size, all_support_counts, min_count = apriori(
    transactions=transactions,
    min_support=DEFAULT_MIN_SUPPORT,
    max_itemset_size=DEFAULT_MAX_ITEMSET_SIZE,
)

frequent_df = frequent_itemsets_to_dataframe(frequent_by_size, len(transactions))
rules_df = generate_association_rules(
    all_support_counts=all_support_counts,
    n_transactions=len(transactions),
    min_confidence=DEFAULT_MIN_CONFIDENCE,
)

frequent_df.to_csv(OUTPUT_DIR / "frequent_itemsets.csv", index=False)
rules_df.to_csv(OUTPUT_DIR / "association_rules.csv", index=False)

print(f"Minimum support count: {min_count}")
print(f"Frequent itemsets: {len(frequent_df)}")
print(f"Association rules: {len(rules_df)}")

## 10. Inspect Results

In [ ]:
print("Top frequent itemsets:")
display(frequent_df.head(20))

print("Top association rules:")
display(rules_df.head(20))

print("Frequent itemset counts by size:")
display(frequent_df.groupby("size").size().reset_index(name="count"))